# 01. Data Ingestion
## Structural Health Monitoring — Heritage Masonry Vaults

**Purpose:** Load monthly raw CSV files exported from the IoT monitoring platform, merge into a unified DataFrame, parse timestamps, and export partitioned subsets by zone and by individual sensor.

**Input:** Monthly CSV files in `../data/raw/` — one file per monitoring month, semicolon-separated, exported from ThingsBoard cloud dashboard.

**Output:**
- `../data/processed/raw_unified.csv` — full unified raw dataset
- `../data/processed/zone_A.csv`, `zone_B.csv`, `zone_C.csv` — partitioned by structural zone
- `../data/processed/sensor_S01.csv` ... `sensor_S12.csv` — partitioned by individual sensor
- `../outputs/tables/00_raw_nan_summary.xlsx` — null value summary table

**Next notebook:** `02_data_cleaning.ipynb`

---
## 0. Sensor & Zone Configuration

Anonymization mapping: original sensor labels (DI-001 to DI-012) replaced with S01–S12.
Structural zones: Zone A (S01–S04), Zone B (S05–S08), Zone C (S09–S12).
Environmental variables: Humidity (Hum.), Atmospheric Pressure (Pres.), Temperature (Temp.).

In [1]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd              # DataFrame handling and manipulation
import numpy as np               # Mathematical functions
import os                        # Directory and path management
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# ============================================================
# SENSOR MAPPING — anonymization dictionary
# Maps original sensor labels to anonymized identifiers
# ============================================================
SENSOR_MAP = {
    'DI-001': 'S01', 'DI-002': 'S02', 'DI-003': 'S03', 'DI-004': 'S04',
    'DI-005': 'S05', 'DI-006': 'S06', 'DI-007': 'S07', 'DI-008': 'S08',
    'DI-009': 'S09', 'DI-010': 'S10', 'DI-011': 'S11', 'DI-012': 'S12'
}

# ============================================================
# ZONE CONFIGURATION
# Defines which sensors belong to each structural zone
# Odd sensors = diagonal axis, Even sensors = horizontal axis
# ============================================================
ZONES = {
    'Zone_A': ['S01', 'S02', 'S03', 'S04'],  # Diagonal: S01, S03 | Horizontal: S02, S04
    'Zone_B': ['S05', 'S06', 'S07', 'S08'],  # Diagonal: S05, S07 | Horizontal: S06, S08
    'Zone_C': ['S09', 'S10', 'S11', 'S12']   # Diagonal: S09, S11 | Horizontal: S10, S12
}

ENV_VARS = ['Hum.', 'Pres.', 'Temp.']        # Environmental variables recorded in parallel
SENSORS  = list(SENSOR_MAP.values())          # List of all anonymized sensor labels

# ============================================================
# PATH CONFIGURATION
# Relative paths — works from any machine without modification
# ============================================================
RAW_DATA_PATH       = '../data/raw/'           # Monthly CSV source files
PROCESSED_DATA_PATH = '../data/processed/'     # Merged and cleaned output CSVs
TABLES_PATH         = '../outputs/tables/'     # Excel summary tables

# Create output directories if they do not exist
for path in [PROCESSED_DATA_PATH, TABLES_PATH]:
    os.makedirs(path, exist_ok=True)

print('Configuration loaded.')
print(f'Sensors: {SENSORS}')
print(f'Zones: {list(ZONES.keys())}')

Configuration loaded.
Sensors: ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12']
Zones: ['Zone_A', 'Zone_B', 'Zone_C']


---
## 1. Load Monthly Raw CSV Files

Each file corresponds to one monitoring month exported from ThingsBoard.
Format: semicolon-separated, columns include Timestamp, DI-001 through DI-012, Hum., Pres., Temp.

Note: March 2025 (250300) was excluded from analysis due to highly irregular readings
during sensor installation and calibration period — not representative of operational behavior.

In [2]:
# ============================================================
# LOAD MONTHLY CSV FILES
# File naming convention: YYMMOO_DistanciaMedida_MM.csv
# where YY=year, MM=month number
# ============================================================

monthly_files = [
    '2504_DisplacementsMeasured_04.csv',
    '2505_DisplacementsMeasured_05.csv',
    '2506_DisplacementsMeasured_06.csv',
    '2507_DisplacementsMeasured_07.csv',
    '2508_DisplacementsMeasured_08.csv',
    '2509_DisplacementsMeasured_09.csv',
    '2510_DisplacementsMeasured_10.csv',
    '2511_DisplacementsMeasured_11.csv',
    '2512_DisplacementsMeasured_12.csv',
    '2601_DisplacementsMeasured_01.csv',
    '2602_DisplacementsMeasured_02.csv'
]

# Load each file and display basic info for verification
dataframes = {}
for file in monthly_files:
    filepath = os.path.join(RAW_DATA_PATH, file)
    df_month = pd.read_csv(filepath, sep=';')
    dataframes[file] = df_month
    print(f'{file}: {df_month.shape[0]} rows, {df_month.shape[1]} columns')

2504_DisplacementsMeasured_04.csv: 346 rows, 16 columns
2505_DisplacementsMeasured_05.csv: 360 rows, 16 columns
2506_DisplacementsMeasured_06.csv: 348 rows, 16 columns
2507_DisplacementsMeasured_07.csv: 360 rows, 16 columns
2508_DisplacementsMeasured_08.csv: 360 rows, 16 columns
2509_DisplacementsMeasured_09.csv: 348 rows, 16 columns
2510_DisplacementsMeasured_10.csv: 321 rows, 16 columns
2511_DisplacementsMeasured_11.csv: 181 rows, 16 columns
2512_DisplacementsMeasured_12.csv: 156 rows, 16 columns
2601_DisplacementsMeasured_01.csv: 360 rows, 16 columns
2602_DisplacementsMeasured_02.csv: 323 rows, 16 columns


---
## 2. Merge All Monthly Files into Unified DataFrame

In [3]:
# ============================================================
# CONCATENATE ALL MONTHLY DATAFRAMES
# ignore_index=True resets the row index after concatenation
# ============================================================
df_raw = pd.concat(list(dataframes.values()), ignore_index=True)

# ============================================================
# TIMESTAMP PARSING
# Convert string timestamps to datetime objects for time-series operations
# ============================================================
df_raw['Timestamp'] = pd.to_datetime(df_raw['Timestamp'], format='%Y-%m-%d %H:%M:%S')

print(f'Unified dataset: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')
print(f'Date range: {df_raw["Timestamp"].min()} to {df_raw["Timestamp"].max()}')
print(f'\nNull values per column:')
print(df_raw.isna().sum())

df_raw.head(5)

Unified dataset: 3463 rows x 16 columns
Date range: 2025-04-01 13:00:00 to 2026-02-27 23:29:30

Null values per column:
Timestamp       0
DI-001        100
DI-002        101
DI-003        284
DI-004        227
DI-005       1077
DI-006       1056
DI-007        634
DI-008        625
DI-009         45
DI-010         39
DI-011         42
DI-012        166
Hum.           38
Pres.          38
Temp.          38
dtype: int64


,Timestamp,DI-001,DI-002,DI-003,DI-004,DI-005,DI-006,DI-007,DI-008,DI-009,DI-010,DI-011,DI-012,Hum.,Pres.,Temp.
0,2025-04-01 13:00:00,12633.000000,24680.600000,12754.000000,24675.000000,11883.714286,24646.727273,11247.000000,24669.230769,12633.125000,24619.925926,12413.923077,24644.833333,45.170000,1013.689000,28.372500
1,2025-04-01 15:00:00,12632.058824,24681.695652,12753.800000,24675.952381,11882.200000,24647.190476,11247.791667,24669.800000,12631.615385,24620.000000,12413.777778,24645.000000,37.040000,1011.568000,33.411500
2,2025-04-01 17:00:00,12633.500000,24682.000000,12751.913043,24676.000000,11880.250000,24647.368421,11248.000000,24671.000000,12628.791667,24619.900000,12411.043478,24645.272727,37.257895,1010.260000,31.130526
3,2025-04-01 19:00:00,12632.714286,24681.956522,12750.352941,24676.636364,11879.000000,24646.750000,11246.259259,24672.000000,12626.814815,24619.888889,12409.416667,24646.000000,41.911111,1009.538889,28.281111
4,2025-04-01 21:00:00,12632.454545,24681.571429,12750.181818,24677.000000,11879.000000,24646.894737,11246.000000,24671.680000,12624.681818,24619.916667,12408.916667,24645.000000,45.882353,1009.838235,26.782353


---
## 3. Anonymize Column Labels

Rename sensor columns from original identifiers (DI-001 through DI-012)
to anonymized labels (S01 through S12) for public publication.

In [4]:
# ============================================================
# RENAME SENSOR COLUMNS — apply anonymization mapping
# ============================================================
df_raw = df_raw.rename(columns=SENSOR_MAP)

print('Column labels after anonymization:')
print(df_raw.columns.tolist())
df_raw.head(3)

Column labels after anonymization:
['Timestamp', 'S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'Hum.', 'Pres.', 'Temp.']


,Timestamp,S01,S02,S03,S04,S05,S06,S07,S08,S09,S10,S11,S12,Hum.,Pres.,Temp.
0,2025-04-01 13:00:00,12633.000000,24680.600000,12754.000000,24675.000000,11883.714286,24646.727273,11247.000000,24669.230769,12633.125000,24619.925926,12413.923077,24644.833333,45.170000,1013.689,28.372500
1,2025-04-01 15:00:00,12632.058824,24681.695652,12753.800000,24675.952381,11882.200000,24647.190476,11247.791667,24669.800000,12631.615385,24620.000000,12413.777778,24645.000000,37.040000,1011.568,33.411500
2,2025-04-01 17:00:00,12633.500000,24682.000000,12751.913043,24676.000000,11880.250000,24647.368421,11248.000000,24671.000000,12628.791667,24619.900000,12411.043478,24645.272727,37.257895,1010.260,31.130526


---
## 4. Export Full Unified Raw Dataset

> **On numeric precision:** All CSV exports in this notebook use `float_format='%.2f'`,
> which writes values to 2 decimal places in the file without modifying the DataFrame
> in memory. Full floating-point precision is preserved for any in-session calculations.
> This is the correct approach — always round at the display/export layer, never on
> the data itself. The 2-decimal resolution is well above sensor measurement precision
> and has no effect on downstream analysis results.

In [5]:
# ============================================================
# EXPORT FULL RAW DATASET
# This is the base file consumed by 02_data_cleaning.ipynb
# ============================================================
# float_format='%.2f' controls decimal display in the exported file
# without modifying the DataFrame in memory — full precision is
# preserved for any in-session calculations.
output_path = os.path.join(PROCESSED_DATA_PATH, 'raw_unified.csv')
df_raw.to_csv(output_path, index=False, float_format='%.2f')
print(f'Full raw dataset exported: {output_path}')
print(f'Shape: {df_raw.shape}')
df_raw.info()

Full raw dataset exported: ../data/processed/raw_unified.csv
Shape: (3463, 16)
<class 'pandas.DataFrame'>
RangeIndex: 3463 entries, 0 to 3462
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Timestamp  3463 non-null   datetime64[us]
 1   S01        3363 non-null   float64       
 2   S02        3362 non-null   float64       
 3   S03        3179 non-null   float64       
 4   S04        3236 non-null   float64       
 5   S05        2386 non-null   float64       
 6   S06        2407 non-null   float64       
 7   S07        2829 non-null   float64       
 8   S08        2838 non-null   float64       
 9   S09        3418 non-null   float64       
 10  S10        3424 non-null   float64       
 11  S11        3421 non-null   float64       
 12  S12        3297 non-null   float64       
 13  Hum.       3425 non-null   float64       
 14  Pres.      3425 non-null   float64       
 15  Temp.      3425 non-n

---
## 5. Partition by Structural Zone

Each zone subset includes its 4 sensors plus the 3 environmental variables.
These subsets are used for zone-level correlation analysis in notebook 04.

In [6]:
# ============================================================
# EXPORT DATA PARTITIONED BY STRUCTURAL ZONE
# Each file contains 4 zone sensors + environmental variables
# ============================================================
for zone_name, zone_sensors in ZONES.items():
    cols = ['Timestamp'] + zone_sensors + ENV_VARS
    df_zone = df_raw[cols].copy()
    output_path = os.path.join(PROCESSED_DATA_PATH, f'{zone_name}.csv')
    # float_format='%.2f' — 2 decimal display, full precision kept in memory
    df_zone.to_csv(output_path, index=False, float_format='%.2f')
    print(f'{zone_name}: {df_zone.shape[0]} rows exported → {output_path}')
    print(f'  Null values: {df_zone.isna().sum().to_dict()}')
    print()

Zone_A: 3463 rows exported → ../data/processed/Zone_A.csv
  Null values: {'Timestamp': 0, 'S01': 100, 'S02': 101, 'S03': 284, 'S04': 227, 'Hum.': 38, 'Pres.': 38, 'Temp.': 38}

Zone_B: 3463 rows exported → ../data/processed/Zone_B.csv
  Null values: {'Timestamp': 0, 'S05': 1077, 'S06': 1056, 'S07': 634, 'S08': 625, 'Hum.': 38, 'Pres.': 38, 'Temp.': 38}

Zone_C: 3463 rows exported → ../data/processed/Zone_C.csv
  Null values: {'Timestamp': 0, 'S09': 45, 'S10': 39, 'S11': 42, 'S12': 166, 'Hum.': 38, 'Pres.': 38, 'Temp.': 38}



---
## 6. Partition by Individual Sensor

Each sensor file contains Timestamp, the individual sensor column, and the 3 environmental variables.
These files are consumed by `02_data_cleaning.ipynb` — one file per sensor.

In [7]:
# ============================================================
# EXPORT DATA PARTITIONED BY INDIVIDUAL SENSOR
# Each file: Timestamp + one sensor + environmental variables
# ============================================================
for sensor in SENSORS:
    cols = ['Timestamp', sensor] + ENV_VARS
    df_sensor = df_raw[cols].copy()
    output_path = os.path.join(PROCESSED_DATA_PATH, f'sensor_{sensor}.csv')
    # float_format='%.2f' — 2 decimal display, full precision kept in memory
    df_sensor.to_csv(output_path, index=False, float_format='%.2f')
    print(f'{sensor}: {df_sensor[sensor].count()} non-null values → {output_path}')

S01: 3363 non-null values → ../data/processed/sensor_S01.csv
S02: 3362 non-null values → ../data/processed/sensor_S02.csv
S03: 3179 non-null values → ../data/processed/sensor_S03.csv
S04: 3236 non-null values → ../data/processed/sensor_S04.csv
S05: 2386 non-null values → ../data/processed/sensor_S05.csv
S06: 2407 non-null values → ../data/processed/sensor_S06.csv
S07: 2829 non-null values → ../data/processed/sensor_S07.csv
S08: 2838 non-null values → ../data/processed/sensor_S08.csv
S09: 3418 non-null values → ../data/processed/sensor_S09.csv
S10: 3424 non-null values → ../data/processed/sensor_S10.csv
S11: 3421 non-null values → ../data/processed/sensor_S11.csv
S12: 3297 non-null values → ../data/processed/sensor_S12.csv


---
## 7. Null Value Summary Table

Generates a summary of null values per sensor vs total timestamp count.
This is the baseline data availability report — before any cleaning is applied.
Exported to Excel for reporting.

In [8]:
# ============================================================
# NULL VALUE SUMMARY TABLE
# Counts recorded vs null values per sensor
# Calculates percentage relative to total timestamp count
# ============================================================
total_timestamps = df_raw['Timestamp'].count()  # Total number of timestamp records

summary_rows = []
for sensor in SENSORS:
    recorded   = df_raw[sensor].count()                         # Non-null values
    nulls      = df_raw[sensor].isna().sum()                    # Null values
    pct_rec    = round(recorded / total_timestamps * 100, 2)    # % of total timestamps
    pct_null   = round(nulls    / total_timestamps * 100, 2)    # % null of total timestamps

    summary_rows.append({
        'Sensor'            : sensor,
        'Recorded Values'   : recorded,
        'Null Values'       : nulls,
        '% Recorded'        : pct_rec,
        '% Null vs Timestamp': pct_null
    })

df_null_summary = pd.DataFrame(summary_rows)

print(f'Total timestamps: {total_timestamps}')
print(f'Date range: {df_raw["Timestamp"].min()} to {df_raw["Timestamp"].max()}')
print()
print(df_null_summary.to_string(index=False))

# Export to Excel
output_path = os.path.join(TABLES_PATH, '00_raw_nan_summary.xlsx')
df_null_summary.to_excel(output_path, index=False)
print(f'\nNull summary table exported: {output_path}')

Total timestamps: 3463
Date range: 2025-04-01 13:00:00 to 2026-02-27 23:29:30

Sensor  Recorded Values  Null Values  % Recorded  % Null vs Timestamp
   S01             3363          100       97.11                 2.89
   S02             3362          101       97.08                 2.92
   S03             3179          284       91.80                 8.20
   S04             3236          227       93.44                 6.56
   S05             2386         1077       68.90                31.10
   S06             2407         1056       69.51                30.49
   S07             2829          634       81.69                18.31
   S08             2838          625       81.95                18.05
   S09             3418           45       98.70                 1.30
   S10             3424           39       98.87                 1.13
   S11             3421           42       98.79                 1.21
   S12             3297          166       95.21                 4.79

Null summa

---
## Summary

This notebook completed the ingestion pipeline:

| Output | Description |
|--------|-------------|
| `raw_unified.csv` | Full unified dataset — all sensors, all months |
| `Zone_A.csv`, `Zone_B.csv`, `Zone_C.csv` | Zone-partitioned subsets |
| `sensor_S01.csv` ... `sensor_S12.csv` | Sensor-partitioned subsets |
| `00_raw_nan_summary.xlsx` | Null value baseline report |

**Next:** `02_data_cleaning.ipynb` — apply geometric filter → Chauvenet Criterion → iterative IQR cleaning per sensor.